In [ ]:
# (setup cell already installs what this notebook needs)

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## Judging by criteria

- A model can score another model's answer against criteria we write
- `labeled_criteria` scores it against a reference answer
- The judge is a model too, with all the same failure modes

Scores one answer for correctness, then lets us change the criteria.

### Exercise Criteria dictionary 
Add additional criteria and score ranges to the list below.

In [1]:
criteria = {
    "correctness": {"desc": "Is the response factually correct?", "scale": (0,5)},
    "conciseness": {"desc": "Is the response concise and on-topic?", "scale": (0,5)},
    "usefulness": {"desc": "Is the response practically useful?", "scale": (0,5)},
}
assert all(k in criteria for k in ["correctness","conciseness","usefulness"])

### Exercise
In the code below, change the criteria parameters (e.g. to "helpfulness") and the prediction and input to our own texts to compare.

In [2]:
from langchain_classic.evaluation import load_evaluator
import json
from dotenv import load_dotenv
load_dotenv()

# 1) We'll use an evaluator with a reference (labeled_criteria)
evaluator = load_evaluator("labeled_criteria", criteria="correctness", llm=make_llm())

# 2) We compare the model's response with the reference
result = evaluator.evaluate_strings(
    prediction="2 + 2 = 4",
    input="Calculate 2 + 2",
    reference="4",
)

print(json.dumps(result, indent=4))

{
    "reasoning": "The criterion for this task is the correctness of the submission. The input asks for the calculation of 2 + 2. The submission provided is 2 + 2 = 4. The reference answer is 4. \n\nComparing the submission to the reference, it is clear that the submission is correct. The calculation of 2 + 2 indeed equals 4, which matches the reference answer. Therefore, the submission meets the criterion of correctness.\n\nY",
    "value": "Y",
    "score": 1
}


### Try judging a wrong answer

- Change the prediction to something plainly false and re-run
- Then make it subtly wrong instead. The obvious error is caught ; the subtle
  one often is not, which is the limit of using a model as the judge